[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NikolaKujoh/chess-game-to-fen/blob/yolo-corner-training/notebooks/02_train_yolo_and_heatmap.ipynb)

# Trening: corner heatmap + YOLO detekcija figura

Trenira dva modela za mobilni server pipeline, na sva tri dostupna skupa sa Drive-a (`chess-project/` folder):
1. **YOLO** - detektuje i klasifikuje figure direktno na originalnoj slici. Trenira se na **chesscog + ChessReD + Roboflow** spojenim u jedan dataset (zamena za occupancy_cnn + piece_cnn par).
2. **corner heatmap** - nalazi 4 ugla table. Trenira se na **ChessReD** (stvarne fotografije) - sintetički model iz originalnog projekta (`models/corner_heatmap.pth`) loše prenosi na stvarne slike (domain gap).

EDA je u `01_dataset_eda.ipynb`. Na kraju se oba modela kopiraju na Google Drive (`chess-project/models/`) za preuzimanje.

## 1. Setup

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(nema GPU - Runtime > Change runtime type > A100)")

Iste putanje na Drive-u kao u prethodnim treninzima (`chess-project/chesscog.zip`, `chessred/`, `annotations.json`, Roboflow zip, i već parsiran `parsed_dataset.json` za chesscog).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/chess-project"
OUT_DIR = f"{DRIVE_ROOT}/models"  # ovde zavrsavaju istrenirani modeli

import os
os.makedirs(OUT_DIR, exist_ok=True)

# chesscog (sintetički, već renderovan)
!cp "$DRIVE_ROOT/chesscog.zip" /content/
!unzip -q /content/chesscog.zip -d /content/chesscog

# ChessReD (stvarne fotografije)
!cp -r "$DRIVE_ROOT/chessred" /content/
!cp "$DRIVE_ROOT/annotations.json" /content/

# Roboflow (dodatni real-world skup, već u YOLO formatu)
!cp "$DRIVE_ROOT/Chess_Pieces.v23-raw.yolov8.zip" /content/
!unzip -q /content/Chess_Pieces.v23-raw.yolov8.zip -d /content/Roboflow

# već parsiran chesscog (spojeni json iz svih train/val/test .json fajlova)
!cp "$DRIVE_ROOT/parsed_dataset.json" /content/

In [ ]:
REPO_URL = "https://github.com/NikolaKujoh/chess-game-to-fen.git"
BRANCH = "yolo-corner-training"  # <-- kad se PR merge-uje u main, ovo se moze izbaciti
!git clone -q -b "$BRANCH" "$REPO_URL" repo 2>/dev/null || (cd repo && git checkout "$BRANCH" -q && git pull -q)
%cd repo/src
!pip install -q -r ../requirements.txt ultralytics pyyaml

## 2. Priprema YOLO dataset-a (chesscog + ChessReD + Roboflow -> jedan combined dataset)

Sva tri skupa se prvo pretvaraju u YOLO format, pa spajaju `merge_yolo_datasets.py`-om. **Napomena o klasama**: ChessReD čuva `category_id` u svom sopstvenom redosledu (bela pa crna: P R N B Q K), koji NIJE isti kao kanonski `PIECE_CLASSES` redosled iz `pipeline.py` (alfabetski) - merge skript to prepoznaje i preslikava po imenu, ne po sirovom indeksu, da klase ne bi ispale pomešane.

In [ ]:
# chesscog -> YOLO (box anotacije koje već postoje u datasetu)
!python export_yolo.py --dataroot /content/chesscog/dataset \
    --records /content/parsed_dataset.json --out /content/yolo_data/chesscog

In [ ]:
# ChessReD -> YOLO, split PO PARTIJI (game_id) da ne bi curelo u test
!python resplit_chessred_by_game.py \
    --annotations /content/annotations.json \
    --dataroot /content/chessred \
    --out /content/yolo_data/chessred_v2 \
    --n-folds 5 --fold 0 --seed 42

In [ ]:
# spoji sva tri u jedan combined dataset (i uskladi redosled klasa)
!python merge_yolo_datasets.py \
    --chesscog /content/yolo_data/chesscog \
    --chessred /content/yolo_data/chessred_v2 \
    --roboflow /content/Roboflow \
    --out /content/yolo_data/combined_v2

## 3. Trening YOLO detektora figura

`batch=64` i `imgsz=640` su ok start za A100 (40/80GB) - povećaj `batch` dalje ako `nvidia-smi` pokaže da ima još slobodne memorije. Ako želiš veću tačnost umesto brzine, probaj `yolov8s.pt` ili `yolov8m.pt` umesto `yolov8n.pt`.

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")
yolo_model.train(
    data="/content/yolo_data/combined_v2/data.yaml",
    epochs=60,
    imgsz=640,
    batch=64,
    device=0,
    project="/content/runs/detect",
    name="chess_pieces_yolo_v2",
)

## 4. Kopiranje YOLO težina na Drive

In [ ]:
import shutil

RUN_DIR = "/content/runs/detect/chess_pieces_yolo_v2/weights"
shutil.copy(f"{RUN_DIR}/best.pt", f"{OUT_DIR}/chess_pieces_yolo_v2_best.pt")
shutil.copy(f"{RUN_DIR}/last.pt", f"{OUT_DIR}/chess_pieces_yolo_v2_last.pt")
print("YOLO sačuvan u:", OUT_DIR)

## 5. Trening corner heatmap modela (na ChessReD)

Isti game-level split (seed/n-folds/fold) kao za YOLO iznad, da test partije ostanu iste za oba modela.

In [ ]:
!python train_corner_chessred.py \
    --annotations /content/annotations.json \
    --dataroot /content/chessred \
    --out /content/models/corner_heatmap_chessred.pth \
    --n-folds 5 --fold 0 --seed 42 \
    --batch-size 32

## 6. Kopiranje corner modela na Drive

In [ ]:
shutil.copy("/content/models/corner_heatmap_chessred.pth",
            f"{OUT_DIR}/corner_heatmap_chessred_fold0.pth")
print("Corner model sačuvan u:", OUT_DIR)

Gotovo. Na Drive-u (`chess-project/models/`) su:
- `chess_pieces_yolo_v2_best.pt` / `_last.pt` - YOLO detektor figura (chesscog+ChessReD+Roboflow)
- `corner_heatmap_chessred_fold0.pth` - corner heatmap za stvarne fotografije

Preuzmi ih odatle - sledeća faza je phone -> PC server sa ova dva modela.